# MMS Stokes — Method of Manufactured Solutions

**Stokes equations on the unit square**: −νΔu + ∇p = f, div u = 0, ν = 1.

**Manufactured solution**:
- Stream function ψ = sin²(πx)sin²(πy) + x²y
- u_x = ∂ψ/∂y = π·sin²(πx)·sin(2πy) + x²
- u_y = −∂ψ/∂x = −π·sin(2πx)·sin²(πy) − 2xy
- p = cos(πx)sin(πy)  (zero mean on [0,1]²)

**Scheme**: Taylor-Hood P2/P1, weak-penalty BC γ = C/h, residual form + Newton,
structured triangular meshes N×N, N = 8, 16, 32, 64, 128.

**Reference rates**: ‖u‖_L2 → 3, ‖u‖_H1 → 2, ‖p‖_L2 → 2 (optimal for TH P2/P1).

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import os, math
import numpy as np
from mpi4py import MPI
import ufl
from petsc4py import PETSc
from dolfinx.mesh import create_unit_square, CellType
from dolfinx.fem import (
    functionspace, Function, Constant, form, assemble_scalar,
)
from dolfinx.fem.petsc import NonlinearProblem
import basix.ufl as bufl


In [ ]:
import os
N_list = [8, 16] if os.environ.get('DD2365_FAST') == '1' else [8, 16, 32, 64, 128]
C_penalty = 1e3


## Manufactured solution

Stream function ψ = sin²(πx)sin²(πy) + x²y

Velocity (divergence-free by construction):
  u_x = π sin²(πx) sin(2πy) + x²
  u_y = −π sin(2πx) sin²(πy) − 2xy

Pressure: p = cos(πx)sin(πy)  (zero mean over [0,1]²)

Body force f = −Δu + ∇p (computed analytically below).

In [ ]:
def u_ex_fn(x):
    pi = np.pi
    ux = pi * np.sin(pi*x[0])**2 * np.sin(2*pi*x[1]) + x[0]**2
    uy = -pi * np.sin(2*pi*x[0]) * np.sin(pi*x[1])**2 - 2*x[0]*x[1]
    return np.vstack([ux, uy])

def p_ex_fn(x):
    return np.cos(np.pi*x[0]) * np.sin(np.pi*x[1])

def f_fn(x):
    """Body force f = -∆u + ∇p."""
    pi = np.pi
    # f_x = -∆u_x + ∂p/∂x
    # ∆u_x = 2π³ sin(2πy)(2cos(2πx)−1) + 2
    lap_ux = 2*pi**3 * np.sin(2*pi*x[1]) * (2*np.cos(2*pi*x[0]) - 1) + 2.0
    dp_dx  = -pi * np.sin(pi*x[0]) * np.sin(pi*x[1])
    fx = -lap_ux + dp_dx
    # f_y = -∆u_y + ∂p/∂y
    # ∆u_y = 2π³ sin(2πx)(4sin²(πy)−1)
    lap_uy = 2*pi**3 * np.sin(2*pi*x[0]) * (4*np.sin(pi*x[1])**2 - 1)
    dp_dy  =  pi * np.cos(pi*x[0]) * np.cos(pi*x[1])
    fy = -lap_uy + dp_dy
    return np.vstack([fx, fy])

# Verify divergence = 0 at a few interior points
import numpy as np
_pts = np.array([[0.3, 0.4, 0.0], [0.7, 0.2, 0.0], [0.5, 0.5, 0.0]])
_u = u_ex_fn(_pts.T)   # shape (2, 3)
h = 1e-7
_dux_dx = (u_ex_fn((_pts + [h,0,0]).T)[0] - u_ex_fn((_pts - [h,0,0]).T)[0]) / (2*h)
_duy_dy = (u_ex_fn((_pts + [0,h,0]).T)[1] - u_ex_fn((_pts - [0,h,0]).T)[1]) / (2*h)
print('Divergence check (max |div u|):', np.max(np.abs(_dux_dx + _duy_dy)))
_pmean = np.mean(p_ex_fn(np.random.uniform(0,1,(3,1000))))
print('Pressure mean check (approx):', abs(_pmean))


## Convergence study  (C = {C_penalty})

Stokes residual form:

    F((u,p);(v,q)) = ∫ν·∇u:∇v dx − ∫p·div(v) dx + ∫div(u)·q dx
                   + γ∫(u−g)·v ds − ∫f·v dx = 0

Solved with NonlinearProblem + NewtonSolver (1 Newton step — linear problem).

In [ ]:
def run_stokes(N_list, C):
    """Stokes MMS convergence study. Returns list of result dicts."""
    results = []
    for N in N_list:
        msh = create_unit_square(MPI.COMM_WORLD, N, N, cell_type=CellType.triangle)

        # Taylor-Hood P2/P1
        P2v = bufl.element('Lagrange', msh.basix_cell(), 2, shape=(2,))
        P1s = bufl.element('Lagrange', msh.basix_cell(), 1)
        TH  = bufl.mixed_element([P2v, P1s])
        W   = functionspace(msh, TH)

        w   = Function(W)
        u, p = ufl.split(w)
        v, q = ufl.split(ufl.TestFunction(W))

        nu    = Constant(msh, PETSc.ScalarType(1.0))
        h_c   = ufl.CellDiameter(msh)
        gamma = Constant(msh, PETSc.ScalarType(C)) / h_c

        # Collapse sub-spaces for interpolation
        Vc, v_map = W.sub(0).collapse()
        Qc, q_map = W.sub(1).collapse()

        g_h = Function(Vc);  g_h.interpolate(u_ex_fn)  # velocity BC
        f_h = Function(Vc);  f_h.interpolate(f_fn)

        # Lift g into mixed space for UFL
        g_W = Function(W)
        g_W.x.array[v_map] = g_h.x.array
        g_W.x.scatter_forward()
        gu, _ = ufl.split(g_W)

        F = (
            nu * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
            - p * ufl.div(v) * ufl.dx
            + ufl.div(u) * q * ufl.dx
            + gamma * ufl.inner(u - gu, v) * ufl.ds
            - ufl.inner(f_h, v) * ufl.dx
        )

        problem = NonlinearProblem(
            F, w,
            petsc_options_prefix='mms_stokes_',
            petsc_options={
                'snes_type': 'newtonls',
                'snes_rtol': 1e-12,
                'snes_atol': 1e-14,
                'ksp_type': 'preonly',
                'pc_type': 'lu',
                'pc_factor_mat_solver_type': 'mumps',
            },
        )
        problem.solve()
        n_iters = problem.solver.getIterationNumber()

        # Extract velocity and pressure
        u_fn = Function(Vc); u_fn.x.array[:] = w.x.array[v_map]
        p_fn = Function(Qc); p_fn.x.array[:] = w.x.array[q_map]

        # Zero-mean correct pressure
        vol    = float(msh.comm.allreduce(assemble_scalar(form(
                    Constant(msh, PETSc.ScalarType(1.0))*ufl.dx)), op=MPI.SUM))
        p_mean = float(msh.comm.allreduce(assemble_scalar(form(p_fn*ufl.dx)), op=MPI.SUM)) / vol
        p_fn.x.array[:] -= p_mean

        # Exact solution functions
        u_ex_h = Function(Vc); u_ex_h.interpolate(u_ex_fn)
        p_ex_h = Function(Qc); p_ex_h.interpolate(p_ex_fn)

        # Error functions
        eu = Function(Vc); eu.x.array[:] = u_fn.x.array - u_ex_h.x.array
        ep = Function(Qc); ep.x.array[:] = p_fn.x.array - p_ex_h.x.array

        e_L2_u = math.sqrt(float(msh.comm.allreduce(
            assemble_scalar(form(ufl.inner(eu, eu)*ufl.dx)), op=MPI.SUM)))
        e_H1_u = math.sqrt(float(msh.comm.allreduce(
            assemble_scalar(form(ufl.inner(ufl.grad(eu), ufl.grad(eu))*ufl.dx)), op=MPI.SUM)))
        e_L2_p = math.sqrt(float(msh.comm.allreduce(
            assemble_scalar(form(ep**2 * ufl.dx)), op=MPI.SUM)))

        h_val = 1.0 / N
        results.append({'N': N, 'h': h_val,
                         'e_L2_u': e_L2_u, 'e_H1_u': e_H1_u, 'e_L2_p': e_L2_p,
                         'iters': n_iters})
        print(f'N={N:4d}  h={h_val:.4f}  ‖u‖_L2={e_L2_u:.4e}  '
              f'‖u‖_H1={e_H1_u:.4e}  ‖p‖_L2={e_L2_p:.4e}  iters={n_iters}')
    return results


In [ ]:
print(f'C = {C_penalty:.0e}')
results_default = run_stokes(N_list, C_penalty)


### Results table (C = {C_penalty})

In [ ]:
def print_stokes_table(results, label=''):
    if label:
        print(f'--- {label} ---')
    hdr = ('N'.rjust(6) + '  ' + 'h'.rjust(7) + '  ' +
           '‖u‖_L2'.rjust(12) + '  ' + 'r_L2u'.rjust(6) + '  ' +
           '‖u‖_H1'.rjust(12) + '  ' + 'r_H1u'.rjust(6) + '  ' +
           '‖p‖_L2'.rjust(12) + '  ' + 'r_L2p'.rjust(6))
    print(hdr)
    for i, r in enumerate(results):
        if i == 0:
            rl2u = rh1u = rl2p = '    —'
        else:
            prev = results[i-1]
            rl2u = f'{math.log2(prev["e_L2_u"]/r["e_L2_u"]):6.2f}'
            rh1u = f'{math.log2(prev["e_H1_u"]/r["e_H1_u"]):6.2f}'
            rl2p = f'{math.log2(prev["e_L2_p"]/r["e_L2_p"]):6.2f}'
        print(f"{r['N']:>6}  {r['h']:>7.4f}  {r['e_L2_u']:>12.4e}  {rl2u}  "
              f"{r['e_H1_u']:>12.4e}  {rh1u}  {r['e_L2_p']:>12.4e}  {rl2p}")

print_stokes_table(results_default, f'C = {C_penalty:.0e}')


### Penalty sensitivity

Re-run at N = {N_list} with C = 1e1 and C = 1e5.

In [ ]:
print('\n=== C = 1e1 ===')
results_c1 = run_stokes(N_list, 1e1)
print('\n=== C = 1e5 ===')
results_c5 = run_stokes(N_list, 1e5)


In [ ]:
print_stokes_table(results_c1, 'C = 1e1')
print()
print_stokes_table(results_c5, 'C = 1e5')
